In [ ]:
#Install necessary packages
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity


In [11]:
#Setting up environment variables
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

foundry_project_endpoint =os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
embedding_model_name = os.getenv("TEXT_EMBEDDING_MODEL_NAME")

Creating the Foundry Project Client

In [12]:
#Setting up the project client
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

Defining Memory Store

In [14]:
from azure.ai.projects.models import MemoryStoreDefaultDefinition, MemoryStoreDefaultOptions

#Define the memory store
definition = MemoryStoreDefaultDefinition(
    chat_model=model_deployment_name,
    embedding_model=embedding_model_name,
    options=MemoryStoreDefaultOptions(
        user_profile_enabled=True,
        chat_summary_enabled=True
    )
)

#Create the memory store
memory_store = project_client.memory_stores.create(
    name="foundry_memory_store_demo_v2",
    definition=definition,
    description="A memory store for storing user profiles and chat summaries and retain user context across sessions."
)

print(f"Created memory store with ID: {memory_store.id} and name: {memory_store.name}")

Created memory store with ID: memstore_6d5a151f3e2464e400KmTPIQ8flGQcl1u0xC4ZJitV8er3e04b and name: foundry_memory_store_demo_v2


Creating a Function to Update the Memory Store

In [18]:
from azure.ai.projects.models import ResponsesUserMessageItemParam

def update_memory_store(memory_store_name:str, user_name:str, user_message:str):
    try:

        #set the scope to associate the memories with
        scope= user_name
        # Create a user message item
        user_message_item = ResponsesUserMessageItemParam(
            content=user_message
        )
        
        update_pollar = project_client.memory_stores.begin_update_memories(
            name=memory_store_name,
            scope=scope,
            items=[user_message_item],
            update_delay=0
        )

        #wait for the update to complete
        update_result = update_pollar.result()

        print(f"Memory store updated with {len(update_result.memory_operations)} memory operations")


        for operation in update_result.memory_operations:
            print(f" - Operation : {operation.kind}, Memory ID: {operation.memory_item.memory_id} content : {operation.memory_item.content}")

        return "updated successfully"
    except Exception as e:
        print(f"Error updating memory store: {str(e)}")
        return "update failed"



In [19]:
update_memory_store(memory_store_name=memory_store.name, user_name="deepanshu", user_message="I am a software developer with 5 years of experience in Python and Azure. I enjoy learning about new technologies and applying them to solve real-world problems., i am also allergic to peanuts and dairy products")

Memory store updated with 5 memory operations
 - Operation : created, Memory ID: ZGVlcGFuc2h1_93b5eebafc7346229e9259732a88429c content : User is a software developer with 5 years of experience.
 - Operation : created, Memory ID: ZGVlcGFuc2h1_5d7431e6333642599edd515efaa9214d content : User specializes in Python and Azure technologies.
 - Operation : created, Memory ID: ZGVlcGFuc2h1_f259982296654531bfc40678a730085f content : User enjoys learning new technologies and applying them to real-world problems.
 - Operation : created, Memory ID: ZGVlcGFuc2h1_e8df173a147049a6888afbeae96ecc64 content : User has allergies to peanuts and dairy products.
 - Operation : created, Memory ID: ZGVlcGFuc2h1_59af63043abe4bda809c0560485a6583 content : The user is a software developer with 5 years of experience specializing in Python and Microsoft Azure technologies. The user enjoys learning new technologies and applying them practically to solve real-world problems. Additionally, the user has allergies to pe

'updated successfully'

Creating the Agent in Foundry Agent Service

In [21]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name= "helpful-assistant-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="you are a helpful assistant. Respond to all queries in a friendly and informative manner."
    ),
)

print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: helpful-assistant-agent:1, name: helpful-assistant-agent, version: 1)


Creating a chat system to see the memories in action

In [23]:
from azure.ai.projects.models import MemorySearchOptions

chat = True

user_name= input("Enter the user name (or type 'exit' to quit): ")

while chat:
    user_message= input("Type \"exit\" to quit the chat, \"update\" to update the memory store with agent response")

    if user_message.lower()=="exit":
        chat = False
    
    #Only update memory store if user entered "UPDATE"
    if user_message.strip().upper()=="UPDATE":
        user_query=input("Enter something that you want to add to the memory store: ")
        update_memory_store(
            memory_store_name=memory_store.name,
            user_name=user_name,
            user_message=user_query
        )
    else:
        #creating the openai client

        openai_client = project_client.get_openai_client()

        #searching for memories and appending to the user message as grounding context

        query_message= ResponsesUserMessageItemParam(content=user_message)

        search_response= project_client.memory_stores.search_memories(
            name= memory_store.name,
            scope= user_name,
            items= [query_message],
            options= MemorySearchOptions(max_memories=5)
        )

        #appending the found memories to the user message to create grounding context
        
        if len(search_response.memories)>0:
            for memory in search_response.memories:
                print(f"found memory: {memory.memory_item.content}")
                user_message += f"\nMemory Context: {memory.memory_item.content}"
        
        response = openai_client.responses.create(
            extra_body={
                "agent":{
                    "name":agent_name,
                    "type":"agent_reference"
                }
            },
            input= user_message
        )

        agent_response= response.output_text

        print(f"Agent: {agent_response}\n")




found memory: The user is a software developer with 5 years of experience specializing in Python and Microsoft Azure technologies. The user enjoys learning new technologies and applying them practically to solve real-world problems. Additionally, the user has allergies to peanuts and dairy products, which is important to consider for any recommendations involving food or related contexts.
found memory: User specializes in Python and Azure technologies.
found memory: User has allergies to peanuts and dairy products.
found memory: User enjoys learning new technologies and applying them to real-world problems.
found memory: User is a software developer with 5 years of experience.
Agent: I'd be happy to share some delicious and easy-to-make recipes you can cook at home! Since you have allergies to peanuts and dairy products, I'll make sure all the recipes are free of those ingredients. Here are some tasty options:

### 1. **Vegan Chickpea Curry**
**Ingredients:**
- 1 can chickpeas (drained